# Bertje Sentence Analysis

## 1. Read data

### 1.1. The dataset of sentences

In [1]:
import pandas as pd
# read the sentence data
sentence_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_data/translated_sentence_data.xlsx")
# check the head
sentence_df.head()

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,0,I was rushed into the [PRESSION-1] [LOCATION-1...
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",1,I was acutely under a painful pressure on the ...
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,2,It started 1 hour after dinner.
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...",3,"I had migraines all day, so I hadn't eaten much."
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...",4,"I got very nauseous, vomiting, dizzy and rejuv..."


In [2]:
# clean sentences as inputs
sentences = sentence_df["Clean_Sentence"].to_list()

### 1.2. Import list of stopwords

In [3]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/mijnidbcoachnlp/data/analysis_data/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg",
    'coach', 'mijnibdcoach'
]

dutch_stopwords.extend(extra_list)

### 1.3 Embed sentences with BERTje

In [4]:
#import the embedding model
from transformers.pipelines import pipeline
from transformers import AutoTokenizer, AutoModel, TFAutoModel
embedding_model = pipeline("feature-extraction", model="GroNLP/bert-base-dutch-cased")

2025-02-07 21:32:19.506032: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
import numpy as np
# load saved bertje sentence-embeddings # we have two different embeddings due to different calculations 
embeddings = np.load("../../data/embeddings/sentence_embeddings_bertje_nl.npy")
#or
#embeddings = np.load("../../data/saved_embeddings/sentence_embeddings_bertje_nl_new.npy") 

In [ ]:
# pre-calculate the sentence embeddings with BERTje if there isn't anything saved 
# calculation 1
import numpy as np

embeddings_nl = embedding_model(sentences, truncation=True, padding=True)
sentence_embeddings_nl = np.array([np.mean(embedding, axis=1).flatten() for embedding in embeddings_nl])
# save the sentence embeddings
import numpy as np
np.save('/workspace/mijnidbcoachnlp/data/saved_embeddings/sentence_embeddings_bertje_nl.npy', sentence_embeddings_nl)

In [ ]:
# calculation two

from tqdm import tqdm 
# Initialize an empty list to store embeddings
sentence_embeddings_nl = []

# Use tqdm to track progress
for sentence in tqdm(sentences, desc="Processing Embeddings", unit="sentence"):
    # Compute embedding for the sentence
    embedding = embedding_model(sentence)
    # Average token embeddings to create a sentence-level embedding
    sentence_embedding = np.mean(embedding[0], axis=0)
    # Append to the list
    sentence_embeddings_nl.append(sentence_embedding)

# Convert the list to a NumPy array
sentence_embeddings_nl = np.array(sentence_embeddings_nl)

# Check the shape of the resulting embeddings
print(f"Shape of sentence embeddings: {sentence_embeddings_nl.shape}")

# save the sentence embeddings
import numpy as np
np.save('/workspace/mijnidbcoachnlp/data/saved_embeddings/sentence_embeddings_bertje_nl_new.npy', sentence_embeddings_nl)

### 1.4 Pre-configure sub-models of BERTopic

In [6]:
# set the vectorizer model
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from hdbscan import HDBSCAN
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired
from umap import UMAP

# configure the sub-models
#vectorizer
vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 2), token_pattern=r'\b[a-zA-Z]{3,}\b')
# clustering (3 options)
hdb_model = HDBSCAN(min_cluster_size=20, min_samples=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
kmeans_model = KMeans(n_clusters=100)
agglo_model = AgglomerativeClustering(n_clusters=100)
# dimension reduction
umap_model=UMAP(n_neighbors=10, n_components=2, metric='cosine', random_state=42)


In [7]:
def build_bertopic(embedding_model, umap_model, cluster_model, vectorizer_model, sentences, embeddings):
    # Initialize BERTopic with specified models
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        top_n_words=10,
        verbose=True
    )

    # Fit-transform to get document topics and probabilities
    topics, probs = topic_model.fit_transform(sentences, embeddings)
    
    # Return only essential results
    return topic_model

## 2. Build/Load Model and Reduce Outliers

### 2.1 Build new model

In [67]:
from bertopic import BERTopic
# build new topic model
topic_model = build_bertopic(embedding_model=embedding_model, umap_model=umap_model, cluster_model=hdb_model, vectorizer_model=vectorizer_model, sentences=sentences, embeddings=embeddings)

2025-02-07 16:28:23,237 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-07 16:29:40,765 - BERTopic - Dimensionality - Completed ✓
2025-02-07 16:29:40,773 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-07 16:29:44,089 - BERTopic - Cluster - Completed ✓
2025-02-07 16:29:44,109 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-07 16:29:45,501 - BERTopic - Representation - Completed ✓


In [92]:
topic_info = topic_model.get_topic_info()
topic_info[topic_info["Representation"].astype(str).str.contains("nutridrink")]

,Topic,Count,Name,Representation,Representative_Docs
303,302,23,302_nutridrink_recept__,"[nutridrink, recept, , , , , , , , ]","[willen jullie a.u.b., Kunnen jullie het recept a.u.b., Is Nutridrink o.i.d.]"


In [ ]:
# examine the topics of the newly built model
topic_model.get_topic_info()

### 2.2 Evaluate the model with coherence score

In [89]:
# get the topics
topics = topic_model.get_topics()
top_words = [[word for word, _ in topic] for topic in topics.values() if topic]  # Top 10 words per topic
top_words

[['afspraak',
  'klachten',
  'keer',
  'weet',
  'last',
  'goed',
  'ontlasting',
  'medicatie',
  'gaan',
  'krijg'],
 ['daags',
  'gebruik',
  'tabletten',
  'prednison',
  'tablet',
  'stuks',
  'gestart',
  'maal',
  'daags tablet',
  'mezavant'],
 ['bekend',
  'uitslag',
  'uitslag ontlasting',
  'uitslagen',
  'bloeduitslagen',
  'uitslagen bloed',
  'ontlasting bekend',
  'bekend uitslag',
  'calprotectine',
  'uitslag calprotectine'],
 ['buikpijn',
  'last',
  'misselijk',
  'pijn',
  'buikkrampen',
  'nacht',
  'buik',
  'gevoel',
  'toilet',
  'diarree'],
 ['ziekenhuis',
  'apotheek ziekenhuis',
  'ziekenhuis apotheek',
  'prikken ziekenhuis',
  'ziekenhuis recept',
  'apotheek',
  'ziekenhuis sturen',
  'afspraak ziekenhuis',
  'zorginstelling',
  'ingeleverd ziekenhuis'],
 ['recept krijgen',
  'krijgen',
  'mogelijk',
  'mogelijk nieuw',
  'hiervoor',
  'ophalen',
  'afspraak inplannen',
  'inplannen',
  'huisarts mogelijk',
  'nieuw recept'],
 ['wilt',
  'nakijken',
  'd

In [97]:
min_words = 10  # Adjust as needed

# Ensure topics have at least `min_words`, removing empty entries and padding if necessary
fixed_topics = []
for topic in top_words:
    if "" in topic:
    # Remove empty elements
        cleaned_topic = [word for word in topic if word]  # Filters out "" and None

        # If topic is too short, repeat the last valid word (if exists)
        if len(cleaned_topic) < min_words and cleaned_topic:
            cleaned_topic += [cleaned_topic[-1]] * (min_words - len(cleaned_topic))

        fixed_topics.append(cleaned_topic)
        print(topic)
        print(cleaned_topic)
    else:
        fixed_topics.append(topic)



['dank voorbaat', 'voorbaat dank', 'voorbaat', 'dank', '', '', '', '', '', '']
['dank voorbaat', 'voorbaat dank', 'voorbaat', 'dank', 'dank', 'dank', 'dank', 'dank', 'dank', 'dank']
['reactie bedankt', 'bedankt reactie', 'reactie', 'bedankt', '', '', '', '', '', '']
['reactie bedankt', 'bedankt reactie', 'reactie', 'bedankt', 'bedankt', 'bedankt', 'bedankt', 'bedankt', 'bedankt', 'bedankt']
['hoor hoor', 'hoor', 'zeggen', 'erbij', 'normaal', '', '', '', '', '']
['hoor hoor', 'hoor', 'zeggen', 'erbij', 'normaal', 'normaal', 'normaal', 'normaal', 'normaal', 'normaal']
['fijne fijne', 'fijne', 'fijne avond', 'avond', 'fijne alvast', 'alvast fijne', 'alvast', '', '', '']
['fijne fijne', 'fijne', 'fijne avond', 'avond', 'fijne alvast', 'alvast fijne', 'alvast', 'alvast', 'alvast', 'alvast']
['ertegen', 'eraan', 'verwachten', '', '', '', '', '', '', '']
['ertegen', 'eraan', 'verwachten', 'verwachten', 'verwachten', 'verwachten', 'verwachten', 'verwachten', 'verwachten', 'verwachten']
['zitte

In [100]:
top_words = fixed_topics
top_words

[['afspraak',
  'klachten',
  'keer',
  'weet',
  'last',
  'goed',
  'ontlasting',
  'medicatie',
  'gaan',
  'krijg'],
 ['daags',
  'gebruik',
  'tabletten',
  'prednison',
  'tablet',
  'stuks',
  'gestart',
  'maal',
  'daags tablet',
  'mezavant'],
 ['bekend',
  'uitslag',
  'uitslag ontlasting',
  'uitslagen',
  'bloeduitslagen',
  'uitslagen bloed',
  'ontlasting bekend',
  'bekend uitslag',
  'calprotectine',
  'uitslag calprotectine'],
 ['buikpijn',
  'last',
  'misselijk',
  'pijn',
  'buikkrampen',
  'nacht',
  'buik',
  'gevoel',
  'toilet',
  'diarree'],
 ['ziekenhuis',
  'apotheek ziekenhuis',
  'ziekenhuis apotheek',
  'prikken ziekenhuis',
  'ziekenhuis recept',
  'apotheek',
  'ziekenhuis sturen',
  'afspraak ziekenhuis',
  'zorginstelling',
  'ingeleverd ziekenhuis'],
 ['recept krijgen',
  'krijgen',
  'mogelijk',
  'mogelijk nieuw',
  'hiervoor',
  'ophalen',
  'afspraak inplannen',
  'inplannen',
  'huisarts mogelijk',
  'nieuw recept'],
 ['wilt',
  'nakijken',
  'd

In [80]:
tokenized_texts = [doc.split() for doc in sentences]

In [103]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Create a dictionary representation of the words in topics
dictionary = Dictionary(tokenized_texts)

# Create a coherence model
coherence_model = CoherenceModel(topics=top_words,
                                 dictionary=dictionary, 
                                 texts = tokenized_texts,
                                 coherence='c_v')

# Compute coherence score
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score: {coherence_score}")

ValueError: unable to interpret topic as either a list of tokens or a list of ids

In [ ]:
# save topic model 
#topic_model.save("/workspace/mijnidbcoachnlp/data/result_data/saved_BERTje_model", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
# save the topic and document information before reducing topics
#save the topic info
#topic_info = topic_model.get_topic_info()
#topic_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/topic_info_bertje.xlsx', index=False)
#save the topic info
#doc_info = topic_model.get_document_info(sentences)
#doc_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/doc_info_bertje.xlsx', index=False)

### 2.2 Reduce the outliers

In [36]:
topic_model = loaded_topic_model

In [15]:
# Reduce outliers
topics = topic_model.topics_
new_topics = topic_model.reduce_outliers(sentences, topics)

100%|██████████| 19/19 [00:03<00:00,  5.48it/s]


In [ ]:
# update topics after reducing outliers
topic_model.update_topics(sentences, topics=new_topics)

In [ ]:
# save the model after reducing outliers
topic_model.save("/workspace/mijnidbcoachnlp/data/result_data/saved_BERTje_model_reduced_outliers", serialization="pytorch", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
# INSPECT topics again # Looks good
topic_model.get_topic_info()

In [30]:
# save the topic and doc info after reducing outliers
topic_model = loaded_topic_model_ro
#save the topic info
topic_info = topic_model.get_topic_info()
topic_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/topic_info_bertje_reduced.xlsx', index=False)
#save the topic info
doc_info = topic_model.get_document_info(sentences)
doc_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/doc_info_bertje_reduced.xlsx', index=False)

### 2.3 Load saved models

In [8]:
from bertopic import BERTopic
# load the model (without reduced outliers)
loaded_topic_model = BERTopic.load("/workspace/mijnidbcoachnlp/saved_models/saved_BERTje_model", embedding_model=embedding_model)

In [9]:
from bertopic import BERTopic
# load the model (without reduced outliers)
loaded_topic_model_ro = BERTopic.load("/workspace/mijnidbcoachnlp/saved_models/saved_BERTje_model_reduced_outliers", embedding_model=embedding_model)

## 3. Visualize the results

In [10]:
topic_model = loaded_topic_model_ro # select the topic model you want to visualize

In [11]:
# settings for plotly
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default = "browser" # option to open the plots in brower

### 3.1 Visualize the hierarchy

In [12]:
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 338/338 [00:02<00:00, 116.36it/s]


### 3.2 Visualize the inter-topic distance map

In [ ]:
# Visualize the inter-topic distance map
topic_model.visualize_topics()

## 4. Merging topics with hierarchy

### 4.1 Merge topics with threshold after outlier-reduction
These results we use in the ECCO abstract

In [13]:
topic_model = loaded_topic_model_ro # use the model before outlier reduction

In [48]:
pd.set_option("display.max_colwidth", None)
hierarchical_topics[hierarchical_topics["Parent_Name"].astype(str).str.contains("medicijnen")] # search for parent cluster to merge based on the hierarchy plot

,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance
268,607,verstandig_pillen_starten_mogelijkheid_medicijnen,"[53, 70, 100, 134, 136, 159, 168, 171, 201, 203, 210, 225, 226, 229, 251, 261, 297, 300, 301, 317, 323, 329, 332]",594,verstandig_mogelijkheid_wachten_roesje_erna,605,pillen_gestart_starten_huidige_doorgaan,1.046576
236,575,huidige_doorgaan_loopt_stap_medicijnen,"[134, 136, 159, 201, 226, 229, 317, 323, 329]",563,huidige_doorgaan_loopt_stap_ver,555,medicijnen_pantoprazol_stelara_extra_zwangerschap,0.988440
216,555,medicijnen_pantoprazol_stelara_extra_zwangerschap,"[134, 136, 229, 329]",500,medicijnen_pantoprazol_stelara_extra_ijzer,513,zwangerschap_vedoluzimab_entocort_bijwerking_kreeg apotheek,0.974478
161,500,medicijnen_pantoprazol_stelara_extra_ijzer,"[134, 229]",134,medicijnen_extra_ijzer_hierbij_schema,229,pantoprazol_stelara_calc_astrazenica_ste,0.930054


In [13]:
topics_to_merge = [[27, 50, 86, 97, 147], #slijm_bloed ontlasting_bloed slijm_ontlasting
                    [0, 6, 32, 105, 286, 334, 326, 61, 209], #last_buikpijn_pijn_toilet_buik	
                    [15, 40, 129, 188, 227, 231, 313, 321, 320, 146, 78, 293, 120, 30, 112, 142, 204, 255], #beter_klachten_gaat beter_klachten klachten_to # this cluster is general health update rather than symptoms
                    [207, 20, 328, 283], # food and weight
                    [103, 106, 135, 62, 63, 113, 288, 14, 81, 88, 92, 109, 149, 43, 248], #medication
                    [154, 197, 244, 302, 133], #vaccin and immunity
                    [222, 246, 262, 280, 220, 213, 176, 66, 257], # advice/discussion
                    [4, 35, 44, 46, 58, 69, 74, 76, 77, 82, 91, 93, 98, 99, 117, 119, 128, 138, 143, 145, 150, 166, 177, 184, 185, 189, 199, 200, 212, 214, 217, 219, 233, 234, 241, 245, 247, 249, 267, 269, 271, 274, 275, 279, 282, 295, 298, 305, 307, 314, 319, 330, 331, 333, 336, 338],
                    [1, 2, 84, 90, 114, 137, 144, 202, 208, 253, 256, 296, 141], # apotheek
                    [26, 157, 178, 221, 263, 316, 48, 94, 107, 115, 179, 205, 264, 73, 290, 24, 59, 158, 318], # forms (admin communication)
                    [17, 304, 327, 64, 118, 194, 55, 127, 11, 18, 33, 57, 122, 265, 235, 191, 294, 83, 10, 13, 87, 110, 123, 164, 174, 223, 292], #contact and appointment
                    [232, 230, 259, 41, 278], #specialist/hospital visit
                    [54, 116, 13], #scans
                    [252, 238, 218, 272, 155, 310, 228, 299, 96, 101, 206, 237, 308, 102], #other
                    [7, 9, 12, 19, 21, 23, 25, 29, 45, 60, 65, 68, 89, 121, 125, 132, 140, 165, 167, 211, 337], #test
                    [28, 37, 51, 183, 190, 193, 266, 134, 136, 229, 210, 108], # medication
                    [329, 215], #pregnancy


]
topic_model.merge_topics(sentences, topics_to_merge)  

In [14]:
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 119/119 [00:00<00:00, 149.68it/s]


In [104]:
pd.set_option("display.max_colwidth", 500)
doc_info = topic_model.get_document_info(docs=sentences)
translated_sentences = sentence_df["Translated_Sentence"]
doc_info["Translated_Sentences"] = translated_sentences
doc_info.loc[doc_info["Topic"] == 16, ["Document", "Representation", "Translated_Sentences"]].sample(10)

,Document,Representation,Translated_Sentences
2319,"/ , (zie mijn bericht van waarop ik terug zou komen als ik op de poli geweest was).","[verhaal, kwijt, ontlastingstest, terugvinden, brief, poli, vraag, vinden, prima verlopen, test]","Hello [PRESSON-1] / [PERSON-2], (see my message from [DATUM-1] on which I would return if I had been on the policy)."
2330,"Lang verhaal, maar goed ;-) Ik hoor graag van de uitkomst.","[verhaal, kwijt, ontlastingstest, terugvinden, brief, poli, vraag, vinden, prima verlopen, test]","Long story, but good;-) I like to hear the outcome."
19132,Ik heb een oproep gekregen voor 12 maart op de poli maar wat gebeurt er tot die tijd ?,"[verhaal, kwijt, ontlastingstest, terugvinden, brief, poli, vraag, vinden, prima verlopen, test]","I got a call for March 12th on the police force, but what happens until then?"
473,"Verder heb ik nog een mail gestuurd naar IBD maar die lun je als niet ontvang beschouwen haha,,ik kwam namelojk ook niet meer in Ibdcoach maar dit is als gelukt.","[verhaal, kwijt, ontlastingstest, terugvinden, brief, poli, vraag, vinden, prima verlopen, test]","Furthermore, I sent an email to IBD but you don't think of that lunch as not receiving haha, I didn't come to Ibdcoach anymore either, but this is a success."
41651,"Dit vind ik wel een beetje vreemd aangezien ik een paar weken geleden, na het per post ontvangen te hebben van de afspraakbevestiging, met de polikliniek heb gebeld met de vraag of er bloed moest worden geprikt.","[verhaal, kwijt, ontlastingstest, terugvinden, brief, poli, vraag, vinden, prima verlopen, test]","I find this a bit strange since a few weeks ago, after receiving the confirmation by post, I called the outpatient clinic asking if blood had to be punctured."
6743,Alles leek wederom prima te verlopen echter bij test B krijg ik wederom iedere keer de melding ''uitleesfout''.,"[verhaal, kwijt, ontlastingstest, terugvinden, brief, poli, vraag, vinden, prima verlopen, test]",Again everything seemed to work out fine with test B however I get the message ''read error' again and again.
27138,Ik heb de app er opnieuw opgezet en de test exact volgens richtlijnen gevolgd.,"[verhaal, kwijt, ontlastingstest, terugvinden, brief, poli, vraag, vinden, prima verlopen, test]",I reset the app and follow the test exactly according to guidelines.
27779,"Heb het zelf vaker van te voren aangegeven bij jullie, maar dan voel ik me zo controlerend.","[verhaal, kwijt, ontlastingstest, terugvinden, brief, poli, vraag, vinden, prima verlopen, test]","I've reported it to you myself before, but then I feel so controlling."
27828,Wat mij betreft is het dan inderdaad prima om de afspraak te verplaatsen naar juli.,"[verhaal, kwijt, ontlastingstest, terugvinden, brief, poli, vraag, vinden, prima verlopen, test]","As far as I am concerned, it is indeed fine to move the appointment to July."
25226,"ik ben , ik ben je mailadres kwijt en ik kan het niet meer terugvinden.","[verhaal, kwijt, ontlastingstest, terugvinden, brief, poli, vraag, vinden, prima verlopen, test]","Hello [PERSOON-1], I am [PERSOON-2] [DATUM-1], I lost your address and I can't find it anymore."


In [102]:
topics_to_merge = [[16], #diet
                    [93, 14, 84, 70], # doctor/hospital visit
                    [19, 40, 29, 56, 55, 110, 72, 24, 51, 92, 104, 97, 59, 6, 50, 114, 38], # general health update
                    [11, 9, 17, 87, 63, 85, 80, 112, 66, 88, 67, 23, 103, 34, 83, 27, 20, 64, 101, 106, 42, 73, 62], # medical advice/discussion
                    [13, 109, 68], # medical imaging examination
                    [8, 7, 81, 119, 54, 52, 36, 65, 41, 82, 76, 49, 32, 102, 74, 57, 116, 107, 28], # Medication and Treatment
                    [45, 44], # Pregnancy
                    [95, 53, 77, 115, 79, 58, 105, 3, 10, 78], # Symptoms
                    [], # Surgery/Operation
                    [15, 100, 48], # Vaccination & Immunity
                    [108, 111, 69], # Work-related Disease Management
                    [5, 29, 96, 39, 31, 117, 33, 117, 89, 118, 98], # Administrative Communication
                    [35, 22, 99, 61, 113, 25], # Apppointment/Contact
                    [18, 12, 4], # Pharmacy
                    [71, 46, 75, 60, 26, 1, 94, 91, 90], # Test Procedure & Results
                    [21, 37, 47, 2, 43, 30] # Other
]
topic_model.merge_topics(sentences, topics_to_merge)  

In [106]:
topics_to_merge = [[16, 8],
                    [-1, 3] # Other
]
topic_model.merge_topics(sentences, topics_to_merge)  

In [107]:
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 14/14 [00:00<00:00, 88.68it/s]


In [108]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,4808,-1_bedankt_dank_reactie_hoor,"[bedankt, dank, reactie, hoor, alvast, hoor graag, snelle, fijn, graag, snelle reactie]","[Alvast bedankt voor de reactie., Alvast bedankt voor de reactie., Ik hoor het graag van u. Bedankt!]"
1,0,5224,0_last_ontlasting_pijn_bloed,"[last, ontlasting, pijn, bloed, buikpijn, slijm, toilet, buik, gevoel, keer]","[Last heb ik altijd eigenlijk., Helaas nu weer veel last veel bloed en slijm en weer 5-8x per dag naar de wc gaan., Sinds 5 dagen heb ik weer slijm bij ontlasting en sinds 3 dagen heb ik weer last van krampen en diarree.]"
2,1,5034,1_gebruik_infuus_humira_medicatie,"[gebruik, infuus, humira, medicatie, crohn, zorgen, daags, zetten, tabletten, medicijnen]","[Gebruik 1xpw humira., Gebruik nu nog 4mg petasa en 5 mg prednison per dag., Ja ik gebruik 2x2 mezant per dag.]"
3,2,4731,2_bloed_prikken_ontlasting_uitslag,"[bloed, prikken, ontlasting, uitslag, laten, laten prikken, bloed laten, bekend, ingeleverd, uitslagen]","[Maandag 2 mei heb ik bloed laten prikken en ontlasting ingeleverd., Ik heb twee weken geleden bloed laten prikken en ontlasting ingeleverd., Ik heb afgelopen vrijdag bloed laten prikken en ontlasting ingeleverd.]"
4,3,4407,3_weet_vraag_advies_hoop,"[weet, vraag, advies, hoop, graag, vakantie, mee, beste, vragen, goed]","[Graag jullie advies., Graag een advies,, Graag advies wat te doen.]"
5,4,4108,4_afspraak_contact_telefonisch_uur,"[afspraak, contact, telefonisch, uur, gebeld, poli, bereikbaar, gepland, ontvangen, staan]","[Dit genoeg tot de afspraak., In maart heb ik een afspraak bij ., Ik heb op een afspraak met .]"
6,5,4025,5_gaat_goed_klachten_voel,"[gaat, goed, klachten, voel, beter, echt, gaat goed, laatste, tijd, moe]","[en , Het gaat niet goed., het gaat toch niet zo goed., Verder gaat het goed met me.]"
7,6,3413,6_recept_apotheek_ziekenhuis_nieuw,"[recept, apotheek, ziekenhuis, nieuw, nieuw recept, sturen, graag, herhaalrecept, zorginstelling, sturen apotheek]","[Dit is mijn apotheek & : T., Mijn apotheerk is apotheek aan de glacisweg., / had ik een nieuw recept voor]"
8,7,3094,7_formulieren_formulier_mogelijk_app,"[formulieren, formulier, mogelijk, app, potje, ontvangen, nieuwe, krijgen, gekregen, oproep]","[Ik heb hier nog geen formulier voor?, Overigens heb ik alleen een formulier voor bloedprikken en ontlasting test voor december gekregen, dus ik neem aan dat ik de formulieren (en potje) voor over 6 maanden nog wel eens krijg., Ik heb namelijk hier geen formulieren meer voor.]"
9,8,737,8_afgesproken_heden_gehoord_vernomen,"[afgesproken, heden, gehoord, vernomen, ontvangen, mailen, uitslag, sturen, afspraak, vandaag]","[Ik heb afgesproken dit te melden i.v.m., Zoals vrijdag afgesproken zou ik nog de datum van de sigmoïdoscopie doorgeven: woensdag 10-11 om 14:35u., Mijn vraag is: wat hadden we afgesproken te doen na dit infuus?]"


In [ ]:
doc_info = topic_model.get_document_info(sentences)
doc_info[doc_info["Topic"] == 301]

In [ ]:

# merge some topics
topics_to_merge = [[2, 12]
]
topic_model.merge_topics(sentences, topics_to_merge)  

In [ ]:
# Manually Put Labels in the Label Column
# Create the customized label column
topic_info["Label"] = None

In [ ]:
topic_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/final_topic_info_bertje.xlsx', index=False)

In [ ]:
labeled_topic_info = pd.read_excel('/workspace/mijnidbcoachnlp/data/result_data/merged_topic_info_bertje_labeled.xlsx')

In [ ]:
labeled_topic_info.rename(columns={"Unnamed: 5": "Model_Label"}, inplace=True)

In [ ]:
merged_doc_info = topic_model.get_document_info(sentences)
merged_doc_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/merged_doc_info_bertje.xlsx', index=False)

In [ ]:
merged_doc_info["Sentence_ID"] = merged_doc_info.index


In [ ]:
annotated_df['Sentence_ID'] = annotated_df['Sentence_ID'].astype(str)
merged_doc_info['Sentence_ID'] = merged_doc_info['Sentence_ID'].astype(str)

In [ ]:
# Now perform the merge
annotated_df = pd.merge(
    annotated_df,
    merged_doc_info[['Sentence_ID', 'Topic', 'Top_n_words']],
    on='Sentence_ID',
    how='left'
)

In [ ]:
annotated_df = pd.merge(
    annotated_df,
    labeled_topic_info[['Topic', 'Model_Label']],
    on='Topic',
    how='left'
)

In [ ]:
# Filter the DataFrame for manual labels equal to "A4"
label = "M7"
test_df = annotated_df[annotated_df["Manual_Label"] == label]

# Calculate the accuracy as the percentage of matches
accuracy = (test_df['Manual_Label'] == test_df['Model_Label']).sum() / len(test_df)

# Output the accuracy
print(f"Accuracy for Manual_Label == {label}: {accuracy * 100:.2f}%")
